### 7.1 处理缺失值   
缺失数据在pandas中呈现的方式有些不完美，但不妨碍真实使用。  
对于float64数据类
型的数据，pandas使用浮点值NaN（NotaNumber）表示缺失数据  。
我们称其为哨兵值，如果存在的话，即说明存在缺失（或空）  值：

In [6]:
import pandas as pd
import numpy as np

In [8]:
float_data = pd.Series([1.2, -3.5, np.nan, 0])
float_data

0    1.2
1   -3.5
2    NaN
3    0.0
dtype: float64

In [10]:
#使用isna方法会生成一个布尔型Series，其中True值表示对应的位置为空值：
float_data.isna()

0    False
1    False
2     True
3    False
dtype: bool

In [12]:
#Python内置的None值也可以作为NA：
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])
string_data

0    aardvark
1         NaN
2        None
3     avocado
dtype: object

In [14]:
string_data.isna()

0    False
1     True
2     True
3    False
dtype: bool

In [16]:
float_data = pd.Series([1, 2, None], dtype='float64')
float_data

0    1.0
1    2.0
2    NaN
dtype: float64

In [18]:
float_data.isna()

0    False
1    False
2     True
dtype: bool

#### Na处理方法

![jupyter](./7.1.png)

#### 7.1.1 过滤缺失值
有多种过滤缺失数据的方法。可以通过pandas.isna或布尔索引的手工方法，但这么做
比较麻烦，更实用的方法是dropna  
。对于Series，dropna返回一个仅含非空数据和索
值的Series：

In [22]:
data = pd.Series([1, np.nan, 3.5, np.nan, 7])
data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

In [24]:
#上述操作等价于
data[data.notna()]    #只会丢弃 单个缺失值，不会按整行丢弃。   drop是整行、列

0    1.0
2    3.5
4    7.0
dtype: float64

In [26]:
#对于DataFrame对象，有多种方式去除缺失值。你可能希望丢弃全NA或含有部分NA的行或列。  
data = pd.DataFrame([[1., 6.5, 3.], [1., np.nan, np.nan],
                    [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])
data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [28]:
#dropna默认丢弃任何含有缺失值的行：
data.dropna()

,0,1,2
0,1.0,6.5,3.0


In [29]:
#传入how=‘all’，只会丢弃全为Na的那些行：
data.dropna(how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


这些返回的都是新的对象，而不是修改原始DF（对象）的内容

In [31]:
#丢弃列 传入axis=‘列’：
data[4] = np.nan
data

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [32]:
data.dropna(axis="columns", how="all")

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [33]:
#只留下含有一定缺失值数量的行，可以用thresh参数实现此目的：
df = pd.DataFrame(np.random.standard_normal((7, 3)))
df.iloc[:4, 1] = np.nan    #前四行第二列赋为空  df[1]是下标为1的列
df.iloc[:2, 2] = np.nan    #前两行第三列赋为空
df

,0,1,2
0,-0.526976,NaN,NaN
1,-0.044675,NaN,NaN
2,-0.578657,NaN,2.030860
3,0.733702,NaN,-0.454051
4,0.012422,0.576774,1.593976
5,-1.205597,-0.166500,1.109940
6,-1.471514,-0.783979,-0.493381


这个修改直接是在df上进行的，永久性修改

In [35]:
df.dropna()

,0,1,2
4,0.012422,0.576774,1.593976
5,-1.205597,-0.166500,1.109940
6,-1.471514,-0.783979,-0.493381


In [38]:
df.dropna(thresh=2)  #保留至少有两个非空的行，保留两个或有两个以上值的行

,0,1,2
2,-0.578657,NaN,2.030860
3,0.733702,NaN,-0.454051
4,0.012422,0.576774,1.593976
5,-1.205597,-0.166500,1.109940
6,-1.471514,-0.783979,-0.493381


In [41]:
df.dropna(axis=1, thresh=5)  #保留存在非空值五个或五个以上的列

,0,2
0,-0.526976,NaN
1,-0.044675,NaN
2,-0.578657,2.030860
3,0.733702,-0.454051
4,0.012422,1.593976
5,-1.205597,1.109940
6,-1.471514,-0.493381


#### 7.1.2 填充空缺值
通过调用fillna,
将缺失值替换为一个常数值：

In [45]:
df.fillna(0)

,0,1,2
0,-0.526976,0.000000,0.000000
1,-0.044675,0.000000,0.000000
2,-0.578657,0.000000,2.030860
3,0.733702,0.000000,-0.454051
4,0.012422,0.576774,1.593976
5,-1.205597,-0.166500,1.109940
6,-1.471514,-0.783979,-0.493381


In [47]:
#若调用fillna时使用字典，就可以实现对不同的列填充不同的值：
df.fillna({1: 0.5, 2: 0})  #第二列Na填充为0.5，第三列Na填充为0

,0,1,2
0,-0.526976,0.500000,0.000000
1,-0.044675,0.500000,0.000000
2,-0.578657,0.500000,2.030860
3,0.733702,0.500000,-0.454051
4,0.012422,0.576774,1.593976
5,-1.205597,-0.166500,1.109940
6,-1.471514,-0.783979,-0.493381


In [49]:
#用于重建索引的插值方法也可以用于fillna：
df = pd.DataFrame(np.random.standard_normal((6, 3)))
df.iloc[2:, 1] = np.nan
df.iloc[4:, 2] = np.nan
df

,0,1,2
0,2.368958,0.376481,1.190340
1,0.394046,0.000152,1.899753
2,1.035593,NaN,-0.521498
3,1.004078,NaN,0.703654
4,1.414834,NaN,NaN
5,-0.668258,NaN,NaN


In [50]:
df.fillna(method="ffill")    #向下填充

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_5224\3944122520.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill")


,0,1,2
0,2.368958,0.376481,1.190340
1,0.394046,0.000152,1.899753
2,1.035593,0.000152,-0.521498
3,1.004078,0.000152,0.703654
4,1.414834,0.000152,0.703654
5,-0.668258,0.000152,0.703654


这个警告信息是告诉你，DataFrame.fillna() 中使用 method 参数（例如 "ffill" 或 "bfill"）的方式即将被弃用，在未来的 pandas 版本中可能会引发错误。

为了避免未来版本的问题，建议使用 ffill() 或 bfill() 方法来替代 fillna() 的 method 参数。你可以根据需要选择使用前向填充或后向填充：  

ffill(): 前向填充（forward fill），会用前一个非空值填充缺失值。  
bfill(): 后向填充（backward fill），会用下一个非空值填充缺失值。  

In [56]:
df.ffill()  #向下填充

,0,1,2
0,2.368958,0.376481,1.190340
1,0.394046,0.000152,1.899753
2,1.035593,0.000152,-0.521498
3,1.004078,0.000152,0.703654
4,1.414834,0.000152,0.703654
5,-0.668258,0.000152,0.703654


In [57]:
df.bfill()  #最后没值，所以无法填充，向上填充

,0,1,2
0,2.368958,0.376481,1.190340
1,0.394046,0.000152,1.899753
2,1.035593,NaN,-0.521498
3,1.004078,NaN,0.703654
4,1.414834,NaN,NaN
5,-0.668258,NaN,NaN


In [58]:
df1 = df
df1.iloc[5,1] = 1
df1

,0,1,2
0,2.368958,0.376481,1.190340
1,0.394046,0.000152,1.899753
2,1.035593,NaN,-0.521498
3,1.004078,NaN,0.703654
4,1.414834,NaN,NaN
5,-0.668258,1.000000,NaN


In [59]:
df1.bfill()  #向前填充

,0,1,2
0,2.368958,0.376481,1.190340
1,0.394046,0.000152,1.899753
2,1.035593,1.000000,-0.521498
3,1.004078,1.000000,0.703654
4,1.414834,1.000000,NaN
5,-0.668258,1.000000,NaN


In [60]:
df.iloc[5,1] = np.nan
df.ffill(limit=2)    #向下填充两个

,0,1,2
0,2.368958,0.376481,1.190340
1,0.394046,0.000152,1.899753
2,1.035593,0.000152,-0.521498
3,1.004078,0.000152,0.703654
4,1.414834,NaN,0.703654
5,-0.668258,NaN,0.703654


df1 = df  
df1.iloc[5,1] = 1    
此时df也跟着变化了，因为df1 = df 并没有创建一个新的 DataFrame，而是让 df1 和 df 指向同一个对象。因此，当你修改 df1 时，实际上是直接修改了 df，因为它们共享相同的内存地址。
  
df1 = df 只是创建了一个新的引用，df1 和 df 都指向了相同的 DataFrame 对象。  
当你执行 df1.iloc[5, 1] = 1 时，实际上是在修改这个共享的对象，因此 df 也会跟着变化。

In [62]:
#如果你想要对 df1 进行修改而不影响 df，需要创建 df 的一个副本。可以使用 copy() 方法来实现：
df1 = df.copy()  # 创建 df 的副本

In [63]:
#利用fillna能实现许多其他功能。例如，可以传人Series的平均值或中位数以替换缺失值：
data = pd.Series([1., np.nan, 3.5, np.nan, 7])
print(data)
data.fillna(data.mean())   #填充平均值

0    1.0
1    NaN
2    3.5
3    NaN
4    7.0
dtype: float64


0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

#### fillna 函数参数说明：

![jupyter](./7.2.png)

### 7.2 数据转化：
除了缺失数据的处理，另一类重要的操作是过滤、清理以及其他
的转换操作。

In [68]:
#### 7.2.1 删除重复数据  
#DataFrame中出现重复行有多种原因。下面是一个例子：
data = pd.DataFrame({"k1": ["one","two"] * 3 + ["two"],
                     "k2": [1, 1, 2, 3, 3, 4, 4]})
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


In [69]:
#DataFrame的duplicated方法返回一个布尔型Series，
#表示各行是不是重复行（列上的值在前面的行中出现过）:
data.duplicated()   #完全重复行

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

duplicated() 方法返回一个布尔值的 Series，用于标记每一行是否为重复行。  
默认情况下，duplicated() 方法会比较所有列，并标记除了第一个出现的值外，后续相同的值为 True。  
第 0 行 (["one", 1]) 不重复，返回 False。  
第 1 行 (["two", 1]) 不重复，返回 False。  
第 2 行 (["one", 2]) 不重复，返回 False。  
第 3 行 (["two", 3]) 不重复，返回 False。  
第 4 行 (["one", 3]) 不重复，返回 False。  
第 5 行 (["two", 4]) 不重复，返回 False。  
第 6 行 (["two", 4]) 和第 5 行完全相同，是重复的，返回 True。  
是一整行中的所有列进行比较，不是某一列，所以只有第六行重复

In [72]:
#还有一个与此相关的drop_duplicates方法，它会返回一个DataFrame，
#其中只保留了duplicated数组中为False的行：(没有重复项)
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


In [74]:
#这两个方法默认会判断全部列，你也可以指定部分列判断是否重复。
#假设我们还有一列值，且希望只根据居“k1”列过滤重复项：
data["v1"] = range(7)   #新建v1列插入7个数
data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [75]:
data.drop_duplicates(subset=["k1"])   #根据k1过滤重复值，默认只保留第一次出现的

,k1,k2,v1
0,one,1,0
1,two,1,1


In [76]:
#duplicated和drop_duplicates默认保留的是第一个出现的值组合。
#传人keep="last"则保留最后一个：
data.drop_duplicates(["k1"], keep="last")

,k1,k2,v1
4,one,3,4
6,two,4,6


In [78]:
data.drop_duplicates(["k1", "k2"], keep="last")  #保留最后出现的，前面出现的都删掉，所以倒数第二行4 5 删去了

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


pandas 中的 duplicated() 和 drop_duplicates() 默认只支持保留第一个或最后一个重复项 (keep="first" 或 keep="last")，并不直接支持保留第几次出现的值组合。  
但是，我们可以通过自定义方法来实现保留某个特定的重复项。  

给重复项分配出现的次数，用 groupby() 和 cumcount() 统计每一行是该组合的第几次出现。  
根据指定的次数进行筛选，从中保留第 n 次出现的值组合。   4


print(result)


In [82]:
# 为每个值组合分配出现的次数
data['occurrence'] = data.groupby(['k1', 'k2']).cumcount() + 1
print(data)

# 筛选出第 2 次出现的组合
result = data[data['occurrence'] == 2]

# 删除辅助列 'occurrence'
result = result.drop(columns='occurrence')
result

    k1  k2  v1  occurrence
0  one   1   0           1
1  two   1   1           1
2  one   2   2           1
3  two   3   3           1
4  one   3   4           1
5  two   4   5           1
6  two   4   6           2


,k1,k2,v1
6,two,4,6


自定义保留第 n 次出现的组合  
如果你希望保留第 3 次出现的组合，只需将 occurrence == 2 改为 occurrence == 3。  
如果某个组合的行数少于你指定的次数，那么这些组合将不会出现在结果中。

#### 7.2.2 利用函数或映射进行数据转换   
对于许多数据集，你可能希望根据某个数组、Series或DataFrame列中的值来实现转换工作

In [85]:
data = pd.DataFrame({"food": ["bacon", "pulled pork", "bacon",
                              "pastrami", "corned beef", "bacon",
                              "pastrami", "honey ham", "nova lox"],
                     "ounces": [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})  #盎司
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,pastrami,6.0
4,corned beef,7.5
5,bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


In [86]:
#添加一列，表示该肉类的动物种类。我们先编写一个不同肉类到动物的映射：
meat_to_animal = {"bacon": "pig",
                  "pulled pork": "pig",
                  "pastrami": "cow",
                  "corned beef": "cow",
                  "honey ham": "pig",
                  "nova lox": "salmon"}

In [87]:
#Series的map方法可以接收一个函数或含有映射关系的字典型对象来实现值转换
data["animal"] = data["food"].map(meat_to_animal)
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


data["animal"] = data["food"].map(meat_to_animal)   
这句话的作用是将 data DataFrame 中 food 列的值，使用字典 meat_to_animal 进行映射，生成一个新的 animal 列。  

map() 方法通过查找 food 列中的值（如 "beef"、"pork" 等）  
并将其替换为字典meat_to_animal 中对应的值（如 "cow"、"pig"）。  
如果 food 列中的某个值没有在字典中找到相应的映射，map() 会返回 NaN。

In [91]:
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,pastrami,6.0,cow
4,corned beef,7.5,cow
5,bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


In [92]:
#可以传入一个能够完成全部工作的函数：
def get_animal(x):
    print(x)
    return meat_to_animal[x]
    
data["food"].map(get_animal)

bacon
pulled pork
bacon
pastrami
corned beef
bacon
pastrami
honey ham
nova lox


0       pig
1       pig
2       pig
3       cow
4       cow
5       pig
6       cow
7       pig
8    salmon
Name: food, dtype: object

print(map(get_animal)) 不能直接使用的原因是：   
data["food"].map(get_animal) 可以正常运行的原因是 pandas 的 map() 方法与 Python 的内置 map() 函数是不同的。  
pandas 的 map() 方法专门用于对 Series 进行元素级的转换，而 Python 的内置 map() 函数则是一个更通用的函数映射工具。

pandas 的 map() 方法：  

用于 pandas.Series，可以接受函数、字典或 Series 进行元素级的转换。  
它会逐个遍历 Series 中的元素，并将每个元素传递给 map() 方法所接收的函数或字典。  

Python 内置的 map() 函数：  

Python 的内置 map() 函数用于将一个函数应用到一个可迭代对象的每个元素上，并返回一个 map 对象（相当于一个迭代器）。  
这个 map 对象是惰性的，需要将其转化为可迭代的类型（如 list）才能看到结果。  

In [98]:
#python内置的map：   
def square(x):
    return x ** 2

#py内置的map()函数将square应用于列表
numbers = [1, 2, 3, 4, 5]
result = map(square, numbers)     #map函数接收一个操作函数，一个可迭代的对象，函数应用于数组中的所有值

#由于map()返回的是一个map对象，需要转化为list才能查看
print(list(result))

[1, 4, 9, 16, 25]


In [100]:
#pandas内置的map：
demo = pd.DataFrame({"numbers": [1, 2, 3, 4, 5]})

def square(x):
    return x ** 2

#使用pd的map方法将square函数应用于 numbers 列
demo["square"] = demo["numbers"].map(square)
demo

,numbers,square
0,1,1
1,2,4
2,3,9
3,4,16
4,5,25


data["animal"] = data["food"].map(meat_to_animal)  
pd中的映射机制：  

map() 是 pandas.Series 的方法：
map() 是 pandas 专门为 Series 对象设计的一个函数，用于将一个函数或字典应用到 Series 中的每个元素。  
它可以根据给定的映射规则（如字典、函数等），将 Series 中的值替换为指定的映射结果。

字典的键值对映射：  
在 pandas.Series.map() 方法中，如果传入的是一个字典，map() 会将 Series 中的每个值视作字典的键，查找该键在字典中对应的值，并将其替换为这个值。  
如果字典中找不到对应的键，pandas 会将该值映射为 NaN。  

对 "apple"：fruit_to_number["apple"] = 1，所以会映射为 1。  
对 "banana"：fruit_to_number["banana"] = 2，所以会映射为 2。  
对 "cherry"：fruit_to_number["cherry"] = 3，所以会映射为 3。  
对 "date"：fruit_to_number["date"] = 4，所以会映射为 4。  


pandas.Series.map() 方法通过遍历每个值，并根据字典中的键-值对进行替换。  
这种映射方式适用于很多场景，尤其是需要将类别变量转换为数值编码的情况。  

In [102]:
#### 7.2.3 替换值
data = pd.Series([1., -999., 2., -999., -1000., 3.])
data

0       1.0
1    -999.0
2       2.0
3    -999.0
4   -1000.0
5       3.0
dtype: float64

In [103]:
# -999 可能是缺失值的标记值，要将其替换为pandas能够理解的Na值，可以用replace生成一个新的Series
data.replace(-999., np.nan)

0       1.0
1       NaN
2       2.0
3       NaN
4   -1000.0
5       3.0
dtype: float64

In [105]:
# 一次性替换多个值，每个值有不同的替换值，可以传入列表
data.replace([-999, -1000], [np.nan, 0])

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

In [106]:
# 传入的参数也可以是字典
data.replace({-999: np.nan, -1000: 0})

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

data.replace方法与data.str.replace不同，后者做的是字符串的元素级替换。  
现在只是对数值替换，没有对字符串替换

#### 7.2.4 重命名轴索引  
与Series中的值一样，轴标签也可以通过函数或者映射进行转换，从而得到一个新的不同的标签对象，也可以就地修改轴，无需新建任何数据结构：

In [110]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])
#轴索引也有map方法
def transform(x):
    return x[:4].upper()  #提取字符串的前四位并大写

data.index.map(transform)     #利用map函数将index中的所有字符串应用到函数中，最后返回的就是每个只有四个字符的列表，一个新的被修改后的索引列表。

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

pd.Series.map() 通常需要一个参数，即应用于每个元素的函数，一般可以对列表中元素进行操作：    
1.自定义函数：传递一个自定义函数，对每个元素进行操作。  
2.字典或 Series：通过映射来替换元素。  
3.pandas.Index.map()：对于 Index 或 Series 对象，这个函数同样适用。

In [111]:
#可以将新生成的 index 赋值给 index的属性，对DF就地修改
data.index = data.index.map(transform)
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


In [112]:
# 如果要创建数据集转换后的版本，并且不修改原始数据，比较实用的方法是rename：
data.rename(index=str.title, columns=str.upper)     #str库，用于处理字符串

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


data.rename(index=str.title, columns=str.upper)   
这句话的作用是对 DataFrame 的行索引（index）和列名（columns）进行重命名：  

index=str.title：将行索引中的每个字符串首字母大写，其余字母小写。  
columns=str.upper：将列名中的每个字符串全部转换为大写。  

In [116]:
# rename可以结合字典型对象实现对部分轴标签的更新：
#部分更新，index更新了ohio， columns更新了three
data.rename(index={"OHIO": "IANDINA"},
            columns={"three": "peekaboo"})

,one,two,peekaboo,four
IANDINA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


如果使用rename，则无须手动复制DataFrame并给index和coLumns属性赋新值。  

rename是创建一个新的DF不会在原DF上做修改  

使用 DataFrame.rename() 方法时，不需要手动创建 DataFrame 的副本并手动为 index 和 columns 赋予新的值。  
rename() 方法可以直接对索引和列名进行修改，并且它的设计使得这个过程更加简洁高效。  

通常，如果不使用 rename() 方法，手动修改 DataFrame 的索引和列名可能需要这么做：
创建副本并手动修改  
data_copy = data.copy()  # 手动创建 DataFrame 的副本  
data_copy.index = [new_index_values]  # 手动给 index 赋新值  
data_copy.columns = [new_column_names]  # 手动给 columns 赋新值  
这种方法需要手动创建一个副本并赋值，这可能显得繁琐且容易出错。

rename() 方法可以让你无需手动复制 DataFrame，也不需要逐一修改 index 和 columns，  
你只需调用该方法并指定修改规则，它会返回一个新的 DataFrame，其中索引和列名已经按照你的要求修改了。  
data_renamed = data.rename(index=str.title, columns=str.upper)

#### 7.2.5 离散化和分箱  
为了便于分析，连续数据常常被离散化或拆分为“箱子”。  
假设有一组人群数据，你希望将其划分为不同的年龄组：  

In [120]:
ages = [1, 20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]
#将这些数据划分为“18到25”“26到35”“35到60”以及“60以上”几个组。要实现该功能，需要使用pandas.cut：
bins = [18, 25, 35, 60, 100]
age_categories = pd.cut(ages, bins)
age_categories

[NaN, (18.0, 25.0], (18.0, 25.0], (18.0, 25.0], (25.0, 35.0], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 13
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

pandas返回的是一个特殊的分类对象。结果展示了pandas.cut划分得到的组。  
age_categories 是一个 pandas.Categorical 对象，显示每个年龄值属于哪个区间。  
比如，年龄 20 被归为 (18, 25] 区间。  
不属于任何区间的值（例如 1）会显示为 NaN。  
Categories 表示定义的四个区间 [18, 25], [25, 35], [35, 60], [60, 100]，表示18到25岁，25到35岁，依此类推。  
默认情况下，区间右闭左开，即 ] 表示右侧闭合。  

In [122]:
#每个组都有特殊（唯一的）区间值类型，包含下限值和上限值：  
age_categories.codes

array([-1,  0,  0,  0,  1,  0,  0,  2,  1,  3,  2,  2,  1], dtype=int8)

age_categories.codes 返回的是 pandas.Categorical 类型对象的编码值。  
每个类别都对应一个整数代码，codes 属性存储这些整数值，表示 Categorical 数据的每个元素所属的类别。  
在 pandas 中，Categorical 是一种处理分类数据的特殊类型。age_categories.codes 会返回每个元素所属类别的编码，而这些编码是从 0 开始的整数。编码的顺序对应于 categories 列表中的类别顺序。

所给的bin中有四类，所以是0~3，-1是在范围外的，输出的数字表明了该数字哪块范围中

In [126]:
age_categories.categories   #查看一共有哪些类别

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [127]:
age_categories.categories[0]    #查看第 1 个类别的内容（范围）

Interval(18, 25, closed='right')

In [128]:
pd.value_counts(age_categories)    #计算每个类别的数量

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_5224\3586575797.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(age_categories)    #计算每个类别的数量


(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

出现 FutureWarning: pandas.value_counts is deprecated and will be removed in a future version 的警告，意味着 pd.value_counts() 函数已经不推荐使用，并将在未来版本中移除。

In [130]:
# 新版本写法
pd.Series(age_categories).value_counts()
pd.DataFrame(age_categories).value_counts()  # 计算每个类别的数量，两种写法都可以

0        
(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

pd.Series 会将 age_categories 转换为一个 Pandas Series 对象，而 value_counts() 是 Pandas Series 上的方法，直接统计每个类别出现的次数。可以直接在 age_categories 上使用 value_counts() 进行频率统计。

In [132]:
#对于区间的字符串表示，圆括号表示边是开放的（不包括），而方括号则表示边是封闭的（包括）。
#哪边是封闭的可以通过right=False进行修改：
pd.cut(ages, bins, right=False)

[NaN, [18.0, 25.0), [18.0, 25.0), [25.0, 35.0), [25.0, 35.0), ..., [25, 35), [60, 100), [35, 60), [35, 60), [25, 35)]
Length: 13
Categories (4, interval[int64, left]): [[18, 25) < [25, 35) < [35, 60) < [60, 100)]

pd.cut() 本身没有直接提供两边都闭的功能，默认是左闭右开，如果两边都闭，需要手动调整。

In [134]:
#通过传递一个列表或数组到labels选项，设置自己的分箱名称：
group_names = ["Youth", "YoungAdult", "MiddleAged", "Senior"]  
pd.cut(ages, bins, labels=group_names)     #bins 定义区间边界，决定属于哪些组

[NaN, 'Youth', 'Youth', 'Youth', 'YoungAdult', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 13
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

pd.cut(ages, bins, labels=group_names) 中的 bins 定义了每个组的边界，而 labels=group_names 则为每个区间分配了相应的标签。  
bins 确定了划分的区间（例如 18-25，25-35，35-60，60-100），它决定每个数值将落入哪个范围。  
labels=group_names 定义了每个区间的名称（标签）。这些标签可以是自定义的字符串，比如 "Youth" 表示 18-25 这个区间，依此类推。

In [195]:
#如果向pandas.cut传入的不是确切的分箱边界，而是分箱的数量，则会根据数据的最小值和最大值计算得到等长的箱。
#将一些均匀分布的数据分成4组：
data = np.random.uniform(size=20)
print(pd.Series.value_counts(data))
numbers = pd.cut(data, 4, precision=2)   #限定小数点后只有两位
numbers.value_counts()

0.104487    1
0.868354    1
0.957414    1
0.806861    1
0.837382    1
0.447675    1
0.903880    1
0.362403    1
0.370552    1
0.404025    1
0.317228    1
0.287515    1
0.524836    1
0.745893    1
0.458738    1
0.338719    1
0.160093    1
0.186883    1
0.700257    1
0.620108    1
Name: count, dtype: int64


(0.1, 0.32]     5
(0.32, 0.53]    7
(0.53, 0.74]    2
(0.74, 0.96]    6
Name: count, dtype: int64

pandas.qcut函数类似于pandas.cut，它可以根据样本分位数对数据进行划分。  
根据数据的分布情况，pandas.cut可能无法使各个分箱中含有相同数量的数据点。  
而由于pandas.qcut使用的是样本分位数，因此可以得到大小基本相等的分箱：

In [197]:
data = np.random.standard_normal(1000)
quartiles = pd.qcut(data, 4, precision=2)
quartiles

[(-0.65, 0.021], (-0.65, 0.021], (0.021, 0.65], (0.021, 0.65], (0.65, 3.26], ..., (0.65, 3.26], (-0.65, 0.021], (0.65, 3.26], (0.65, 3.26], (-3.34, -0.65]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.34, -0.65] < (-0.65, 0.021] < (0.021, 0.65] < (0.65, 3.26]]

In [199]:
#旧版写法：pd.value_counts(quartiles)
quartiles.value_counts()

(-3.34, -0.65]    250
(-0.65, 0.021]    250
(0.021, 0.65]     250
(0.65, 3.26]      250
Name: count, dtype: int64

In [142]:
#与pandas.cut类似，你也可以传递自定义的分位数（0到1之间的数值，包含端点）：
pd.qcut(data, [0, 0.1, 0.5, 0.9, 1.]).value_counts()

(-3.3979999999999997, -1.284]    100
(-1.284, -0.0482]                400
(-0.0482, 1.357]                 400
(1.357, 3.108]                   100
Name: count, dtype: int64

数据按指定的分位数进行分箱的函数  


[0, 0.1, 0.5, 0.9, 1.]: 这是分位点，表示你想要将数据分为 4 个区间：  
第一个区间是前 10% 的数据 (0 到 0.1)，  
第二个区间是从 10% 到 50% (0.1 到 0.5)，  
第三个区间是从 50% 到 90% (0.5 到 0.9)，  
最后一个区间是从 90% 到 100% (0.9 到 1)。  
所以一共1000个随机正态分布数，划分为10份，一区间100， 二区间400，以此类推，等频划分

#### 区别：
pd.cut()（等宽分箱）  
pd.qcut()（等频分箱）   
pd.cut()：固定区间宽度，但每个区间内的数据点数量可能不同。  
pd.qcut()：区间宽度不固定，但每个区间的数据点数量相等。

 



#### 7.2.6 检测和过滤异常值
过滤或转换异常值在很大程度上就是运用数组运算。

In [207]:
data = pd.DataFrame(np.random.standard_normal((1000, 4)))
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.006944,-0.039591,0.044071,-0.049200
std,1.008725,1.007562,0.993426,0.979236
min,-3.147019,-4.148521,-3.280524,-3.325611
25%,-0.637697,-0.730841,-0.668410,-0.736051
50%,0.034101,-0.074870,0.000929,-0.040612
75%,0.713286,0.692676,0.760569,0.614423
max,2.836040,3.212068,3.270993,3.577423


In [209]:
#找出某列中绝对值超过3的值：
col = data[2]
col[col.abs() > 3]

174    3.093334
190    3.035316
626   -3.280524
741    3.270993
948   -3.261867
Name: 2, dtype: float64

In [211]:
#要选出全部含有“超过3或者-3的值”的行，可以在Bool型DF中使用any方法
data[(data.abs() > 3).any(axis="columns")]   #只要列中有绝对值大于3的就输出该行

,0,1,2,3
57,-0.637902,-0.512654,-0.907969,-3.153711
140,0.947240,-0.081779,-0.255104,3.577423
174,-0.333175,3.103907,3.093334,0.864345
190,1.296396,-0.726887,3.035316,0.749068
249,0.626529,-3.176571,-0.005608,0.411642
296,-0.352726,-0.148074,1.873258,-3.325611
494,-3.147019,-0.218694,-0.583157,-0.167009
626,-1.761959,1.696907,-3.280524,1.096799
639,-0.011942,3.212068,-0.629072,-0.622271
650,-0.899043,-3.342905,-0.438670,2.375037


人话：遍历1000 * 4的矩阵中是否有满足条件的项，满足则输出  

any() 是 Python 的一个内置函数，用于判断一个可迭代对象（如列表、元组、集合等）中是否至少有一个元素为真。  
只要其中有一个元素为 True，any() 函数就返回 True；否则，返回 False。如果可迭代对象为空，any() 返回 False。

data.abs（）>3外面的括号是必需的，这是因为要对比较的结果调用any方法。  


In [223]:
# 对绝对值大于3的进行操作   先生成bool数组然后再进行操做
data[data.abs() > 3] = np.sign(data) * 3   #将data中绝对值大于3的根据符号去赋值3还是-3
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.007091,-0.038239,0.044214,-0.049298
std,1.008275,1.000922,0.990451,0.975748
min,-3.000000,-3.000000,-3.000000,-3.000000
25%,-0.637697,-0.730841,-0.668410,-0.736051
50%,0.034101,-0.074870,0.000929,-0.040612
75%,0.713286,0.692676,0.760569,0.614423
max,2.836040,3.000000,3.000000,3.000000


numpy.sign() 是 NumPy 库中的一个函数，用来返回输入数组中每个元素的符号：  

如果元素是正数，返回 1  
如果元素是负数，返回 -1  
如果元素是零，返回 0  

In [225]:
#根据data的值是正还是负，np.sign（data）可以生成1和-1：
np.sign(data).head()

,0,1,2,3
0,-1.0,-1.0,1.0,-1.0
1,-1.0,-1.0,-1.0,-1.0
2,-1.0,1.0,-1.0,1.0
3,-1.0,1.0,-1.0,-1.0
4,1.0,1.0,-1.0,1.0


#### 7.2.7 置换和随机采样  
利用numpy.random.permutation函数可以轻松实现对Series或DataFrame列的置换（即随机重排序）。  
通过对需要排列的轴的长度调用permutation，可产生一个表示新顺序的整数数组：

In [232]:
df = pd.DataFrame(np.arange(5 * 7).reshape(5, 7))    #reshape形状
df

,0,1,2,3,4,5,6
0,0,1,2,3,4,5,6
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
3,21,22,23,24,25,26,27
4,28,29,30,31,32,33,34


In [238]:
#permutation 翻译为排列
sampler = np.random.permutation(5)   #生成一个包含 0 到 4 的随机排列数组
sampler

array([3, 1, 2, 4, 0])

In [244]:
#该数组可以用于基于iloc的索引操作或等价的take函数：
df.take(sampler)   #根据sampler中的顺序对df中的行进行抽取

,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
4,28,29,30,31,32,33,34
0,0,1,2,3,4,5,6


take 函数用于根据指定的行或列位置索引从 DataFrame 或 Series 中选择数据。  
如果超出DF范围则无法操作

In [246]:
df.iloc[sampler]

,0,1,2,3,4,5,6
3,21,22,23,24,25,26,27
1,7,8,9,10,11,12,13
2,14,15,16,17,18,19,20
4,28,29,30,31,32,33,34
0,0,1,2,3,4,5,6


In [253]:
# 在take函数中使用参数axis="columns”，还可以对于列进行置换：
columns_sampler = np.random.permutation(7)
columns_sampler

array([4, 6, 1, 5, 2, 0, 3])

In [251]:
df.take(columns_sampler, axis="columns")

,5,1,6,3,0,4,2
0,5,1,6,3,0,4,2
1,12,8,13,10,7,11,9
2,19,15,20,17,14,18,16
3,26,22,27,24,21,25,23
4,33,29,34,31,28,32,30


In [262]:
# iloc 改变列     iloc左部分为行操作，右部分为列操作
df.iloc[:, columns_sampler]   #选取所有行，列按照columns_sampler中的顺序抽取列

,4,6,1,5,2,0,3
0,4,6,1,5,2,0,3
1,11,13,8,12,9,7,10
2,18,20,15,19,16,14,17
3,25,27,22,26,23,21,24
4,32,34,29,33,30,28,31


In [270]:
#如果不想用替换的方式选取随机子集，可以在Series和DataFrame上使用sample方法：
df.sample(n=3)   #从行中抽取三个随机样本

,0,1,2,3,4,5,6
2,14,15,16,17,18,19,20
0,0,1,2,3,4,5,6
4,28,29,30,31,32,33,34


sample 是 Pd 函数，用于从 DF 或 Series 中随机抽取指定数量的样本。 

它可以用于数据的随机抽样、数据分割和其他需要随机选择的情况。  

DataFrame.sample(n=None, frac=None, replace=False, weights=None, random_state=None, axis=None)  
n：要抽取的样本数量。如果指定了 n，则 frac 参数必须为 None。  
frac：要抽取的样本比例。如果指定了 frac，则 n 参数必须为 None。例如，frac=0.1 表示抽取 10% 的样本。  
replace：是否允许重复抽样。False 表示不允许重复，True 表示允许重复。  
weights：样本的权重。可以是一个与 DataFrame 或 Series 长度相同的数组，表示每个样本被抽中的概率。  
random_state：用于设置随机种子，以便在多次运行时获得相同的结果。可以是整数或 np.random.RandomState 实例。  
axis：选择抽样的轴。0 表示按行抽样，1 表示按列抽样。默认是 None，即按行抽样。


In [272]:
#要通过替换的方式生成样本（允许重复选择），可以在sample中传人replace=True：
choices = pd.Series([5, 7, -1, 6, 4])   
choices.sample(n=10, replace=True)    #从choices中抽10条，可以重复

1    7
0    5
3    6
4    4
2   -1
3    6
4    4
2   -1
3    6
2   -1
dtype: int64

#### 7.2.8 计算指标/虚拟变量   
另一种常用于统计建模或机器学习的转换方式是将分类变量转换为“虚拟”或“指标矩阵”。  
如果DataFrame的某一列中含有k个不同的值，则可以派生出一个k列矩阵或DF（其值全为1和0）。pandas有一个pandas.get_dummies函数可以实现该功能。

In [277]:
df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                   "data1": range(6)})
df

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [289]:
pd.get_dummies(df["key"]).astype(int)   #将key列转化为虚拟变量

,a,b,c
0,0,1,0
1,0,1,0
2,1,0,0
3,0,0,1
4,1,0,0
5,0,1,0


get_dummies 函数将 key 列中的每个唯一值（'a', 'b', 'c'）转换为一个新的二进制虚拟变量列，表示每个样本是否具有这些值。结果是一个新的 DataFrame，其中每一列代表 key 列中一个唯一值的虚拟变量，值为 0 或 1。

In [ ]:
pd.get_dummies(df["key"])   
   a  b  c  
0  0  1  0  
1  0  1  0  
2  1  0  0  
3  0  0  1  
4  1  0  0  
5  0  1  0  
每一列代表 key 列中的一个唯一值：a, b, 和 c。   
每行的值：在每行中，虚拟变量列 a, b, 和 c 的值是 0 或 1，表示原始 key 列在该行的值是否与该列名称相匹配。  
第 0 行中，key 列的值是 'b'，因此虚拟变量列 b 的值为 1，其他列的值为 0。  
第 2 行中，key 列的值是 'a'，因此虚拟变量列 a 的值为 1，其他列的值为 0。  
转化后这些二进制变量可用于机器学习和数据分析，这种转换是处理分类数据的标准做法，便于模型处理这些数据

In [295]:
# 给指标DF的列加上一个前缀，以便能够与其他数据进行合并。pandas.get_dummies的prefix参数可以实现该功能：
dummies = pd.get_dummies(df["key"], prefix="key")
print(dummies)
df_with_dummy = df[["data1"]].join(dummies)  #data1在前面"data1": range(6)，join第八章有说明
df_with_dummy.astype(int)

   key_a  key_b  key_c
0  False   True  False
1  False   True  False
2   True  False  False
3  False  False   True
4   True  False  False
5  False   True  False


,data1,key_a,key_b,key_c
0,0,0,1,0
1,1,0,1,0
2,2,1,0,0
3,3,0,0,1
4,4,1,0,0
5,5,0,1,0


In [297]:
#如果DataFrame中的某行属于多个分类，我们就必须使用另一种方法创建虚拟变量。
#观察MovieLens 1M数据集，第13章会更深人地研究该数据集：
mnames = ["movie_id", "title", "genres"]
movies = pd.read_table("../datasets/movielens/movies.dat", sep="::",
                       header=None, names=mnames, engine="python")
movies[:10]

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children's
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [299]:
#pandas实现了一个特殊的Series方法str.get_dummies（7.4节会详细讨论以str.开头的方法）  
#可以处理这种多个组成员编码为分隔字符串的场景：
dummies = movies["genres"].str.get_dummies("|")     #将genres列中的所有数据拆分，并通过创建虚拟变量的方法清晰的展示了每行都有哪些种类的genres（流派）
dummies.iloc[:10, :6]  

,Action,Adventure,Animation,Children's,Comedy,Crime
0,0,0,1,1,1,0
1,0,1,0,1,0,0
2,0,0,0,0,1,0
3,0,0,0,0,1,0
4,0,0,0,0,1,0
5,1,0,0,0,0,1
6,0,0,0,0,1,0
7,0,1,0,1,0,0
8,1,0,0,0,0,0
9,1,1,0,0,0,0


In [303]:
#使用add_prefix方法给dummies的列名添加前缀“Genre_”
#将dummies和movies连接起来
movies_windic = movies.join(dummies.add_prefix("Genre_"))
movies_windic.iloc[0]  #输出第0行，也就是header

movie_id                                       1
title                           Toy Story (1995)
genres               Animation|Children's|Comedy
Genre_Action                                   0
Genre_Adventure                                0
Genre_Animation                                1
Genre_Children's                               1
Genre_Comedy                                   1
Genre_Crime                                    0
Genre_Documentary                              0
Genre_Drama                                    0
Genre_Fantasy                                  0
Genre_Film-Noir                                0
Genre_Horror                                   0
Genre_Musical                                  0
Genre_Mystery                                  0
Genre_Romance                                  0
Genre_Sci-Fi                                   0
Genre_Thriller                                 0
Genre_War                                      0
Genre_Western       

对于更大的数据，用这种方式创建多成员指标变量没有那么快。最好使用更
底层的函数，将其直接写入NumPy数组，然后将结果封装在DataFrame中。

In [310]:
#一个在统计应用中的诀窍，是将pandas.get_dummies和pandas.cut等离散化函数结合起来使用：
np.random.seed(12345)
values = np.random.uniform(size=10)  #默认生成[0,1)的浮点数，可以通过设置去改变区间和数据类型
values

array([0.92961609, 0.31637555, 0.18391881, 0.20456028, 0.56772503,
       0.5955447 , 0.96451452, 0.6531771 , 0.74890664, 0.65356987])

In [308]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

In [312]:
pd.get_dummies(pd.cut(values, bins))

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,True
1,False,True,False,False,False
2,True,False,False,False,False
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,True,False,False
6,False,False,False,False,True
7,False,False,False,True,False
8,False,False,False,True,False
9,False,False,False,True,False


### 7.3 扩展数据类型  
一般情况下pd会把数据类型转化为float64，并使用np.nan填充空缺值

In [317]:
s = pd.Series([1, 2, 3, None])
s

0    1.0
1    2.0
2    3.0
3    NaN
dtype: float64

In [319]:
s.dtype

dtype('float64')

因为要考虑到向后兼容，Series继承使用fLoat64数据类型和np.nan表示缺失数据。  
我们还可以使用pandas.Int64Dtype来创建这个Series：

In [324]:
s = pd.Series([1, 2, 3, None], dtype=pd.Int64Dtype())
s

0       1
1       2
2       3
3    <NA>
dtype: Int64

In [328]:
s.isna()

0    False
1    False
2    False
3     True
dtype: bool

In [331]:
s.dtype

Int64Dtype()

In [335]:
# 输出<NA>指明扩展类型数组中存在缺失值。这里使用的是特殊的pandas.NA作为哨兵值：
s[3]

<NA>

In [337]:
s[3] is pd.NA

True

In [343]:
#我们还可以使用缩写“Int64”替代pd.Int64Dtype()来指明数据类型。
#首字母大写是必需的，否则就成了NumPy的非扩展类型：
s = pd.Series([1, 2, 3, None], dtype="Int64") 
s.dtype

Int64Dtype()

In [345]:
#pandas还有一种扩展类型，专门用于未使用NumPy对象数组的字符串数据（需要安装pyarrow库）：
s = pd.Series(['one', 'two', 'None', 'three'], dtype=pd.StringDtype())
s

0      one
1      two
2     None
3    three
dtype: string

此类字符串数组通常使用更少的内存，通常在大型数据集上的运算更为高效.  

pd的扩展数据类型  

![jupyter](7.3.png)

### 7.4 字符串操作  
#### 7.4.1 py内置的字符串对象方法：  
py内置的字符串处理和脚本应用，大部分都有：

In [358]:
#以逗号为分隔符的字符串可以用split拆分成数个段
val = "a,b, guido"
val.split(",")

['a', 'b', ' guido']

In [360]:
# split 常与 strip 使用，去除空格符号，包括换行符
pieces = [x.strip() for x in val.split(",")]
pieces

['a', 'b', 'guido']

In [362]:
#利用加法，可以将这些子字符串以双冒号分隔符的形式连接起来：
first, second, third = pieces
first + "::" + second + "::" + third

'a::b::guido'

In [ ]:
如果你有一个包含多个元素的列表（例如 ['a', 'b', 'guido']）   
可以使用多重赋值的方式将列表中的元素一一对应地赋值给多个变量。  
这种操作称为“拆包”或“解包”。  
pieces = ['a', 'b', 'guido']  
first, second, third = pieces  

print(first)   # 输出: 'a'  
print(second)  # 输出: 'b'  
print(third)   # 输出: 'guido'  

利用*rest捕捉剩余的元素      
pieces = ['a', 'b', 'guido', 'python']      
first, second, *rest = piece      s

print(fir  st)   # 输出:     'a'
print(sec  ond)  # 输出:     'b'
print(r  est)    # 输出: ['guido', 'pyt    hon']


In [370]:
#这种方式并不是很实用。
#更快且更符合Python风格的方式是对字符串"：："的join方法传入列表或元组：
"::".join(pieces)

'a::b::guido'

In [374]:
# 子字符串定位： in、 index、find
"guido" in val

True

In [378]:
val.index("guido")

5

In [380]:
val.find("guido")

5

In [384]:
# index 与 find 的区别：
val.index("abc")  #找不到会抛出异常

ValueError: substring not found

In [388]:
val.find("abc")  #返回 -1

-1

In [390]:
#统计子串出现次数：
val.count(",")

2

In [392]:
# replace用于将指定模式字符串替换为另一种。通过传入空字符串，它也常常用于删除特定模式：
val.replace(",", "::")

'a::b:: guido'

In [403]:
val.replace(",", "")

'ab guido'

In [400]:
val.replace([",", " "], "::")   

TypeError: replace() argument 1 must be str, not list

replace() 函数只能一次替换一个字符或字符串，不能直接传入一个列表或多个字符进行替换。  
多个替换用正则表达式  

#### py内置字符串方法：  

![jupyter](7.4.png)

### 7.4.2 正则表达式  
re模块的函数可以分为三个大类：模式匹配、替换以及拆分。   ）。

In [414]:
#拆分一个字符串，分隔符为数量不定的一组空白符（制表符、空格、换行符等）。
#描述一个或多个空白符的正则表达式 \s+:
import re
text = "foo     bar\t baz   \tqux"
re.split(r"\s+", text)

['foo', 'bar', 'baz', 'qux']

调用re.split（"\s+"，text）时，正则表达式会先被编译，然后再在text上调用其split方法。  
可以用re.compile自已编译正则表达式，以得到可重复使用的正则表达式对象：

In [432]:
# 将多个空白正则表达式定义为 regex，方便后期调用，如果多次使用，则强烈推荐
regex = re.compile(r"\s+")    #re是pd方法，compile用来将r"\s+"编译为一个正则表达式对象。这个对象可以用于多次匹配操作。
regex.split(text)

['foo', 'bar', 'baz', 'qux']

In [424]:
#获取所有的
regex.findall(text)

['     ', '\t ', '   \t']

regex.findall(text) 返回的是一个列表，其中包含了所有与正则表达式 r"\s+" 匹配的空白字符序列。  

r"\s+" 匹配一个或多个空白字符。  
这个正则表达式模式会匹配文本中的空白字符序列（如空格、制表符、换行符等）  
所以最后输出为提取的空格、制表符、换行符等

如果想避免正则表达式中不需要的转义(\)，则可以使用原生字符串语法，  
如r"C\x”（而非等价的"C\\x"）。

原生字符串（raw strings）是 Python 提供的一种字符串表示方式，它以 r 或 R 开头，告诉 Python 解释器不要对字符串中的反斜杠进行转义。使用原生字符串语法可以让你更方便地书写包含反斜杠的正则表达式。  

match和search与findall功能类似。  
findall返回的是字符串中所有的匹配项，而search则只返回第一个匹配项。  
match更加严格，它只匹配字符串的起始位置。  ：

In [479]:
# 识别大部分电子邮件地址的正则表达式：
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com"""
pattern = r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}"

# regex 是一个正则表达式对象，可以多次使用。
# re.IGNORECASE使正则表达式不区分大小写
regex = re.compile(pattern, flags=re.IGNORECASE)

#使用findall得到一组电子邮件地址：
regex.findall(text)

['dave@google.com', 'steve@gmail.com', 'rob@gmail.com', 'ryan@yahoo.com']

In [ ]:
[A-Z0-9._%+-]+：  

[A-Z0-9._%+-] 是一个字符集，匹配大写字母（A-Z）、数字（0-9）、点号（.）、下划线（_）、百分号（%）、加号（+）和减号（-）。
+ 表示一个或多个字符的组合，即电子邮件的用户名部分可以包含这些字符，且至少需要一个字符。  
@：  

字面上的 @ 符号，表示电子邮件地址中的分隔符。
[A-Z0-9.-]+：

[A-Z0-9.-] 是一个字符集，匹配大写字母、数字、点号（.）和连字符（-）。
+ 表示一个或多个字符的组合，即电子邮件的域名部分可以包含这些字符，且至少需要一个字符。
\.：

字面上的点号（.），用来分隔域名和顶级域名。
[A-Z]{2,4}：

[A-Z] 是一个字符集，匹配大写字母。
{2,4} 表示匹配长度在 2 到 4 个字符之间的顶级域名（如 .com、.org、.net 等）。

In [481]:
m = regex.search(text)  #通过自己设定的正则表达式找到第一个匹配的字符串
m

<re.Match object; span=(5, 20), match='dave@google.com'>

search返回的是文本中第一个电子邮件地址（以特殊的匹配对象形式返回）。  
对于前面提到的正则表达式，匹配对象只能告诉我们模式在原字符串中的起始位置和结束位置：

匹配的电子邮件地址：dave@google.com 是在文本中找到的第一个匹配项。  
匹配的位置：(5, 20) 表示 dave@google.com 在原字符串中的起始位置是 5，结束位置是 20（不包括 20）。  

In [483]:
text[m.start():m.end()]

'dave@google.com'

In [485]:
print(regex.match(text))

None


match()是用来在字符串 text 的开始位置检查是否符合正则表达式 regex 的方法。

如果正则表达式从字符串的起始位置开始匹配成功，match() 方法返回一个 Match 对象  
如果不匹配或不从字符串的起始位置匹配，返回 None。

因为第一个Dave dave@google.com，是Dave，中间有空格，并不符合自己设定的正则表达式，所以match返回为空  
若改"""dave@google.com""",则match返回  
匹配的电子邮件地址: dave@google.com   
匹配的位置: (0, 15)  


In [487]:
# sub方法可以将匹配到的模式替换为指定字符串，并返回所得到的新字符串:
print(regex.sub("REDACTED", text))    #所有符合正则表达式的邮件地址都换为这个字符串了

Dave REDACTED
Steve REDACTED
Rob REDACTED
Ryan REDACTED


In [489]:
#不仅想找出电子邮件地址，还想将各个地址分成三个部分：用户名、域名以及域名后缀。
#要实现此功能，只需将待分割的模式的各部分用圆括号括起来即可：
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"     #每个要输出的部分用圆括号括起来即可
regex = re.compile(pattern, flags=re.IGNORECASE)

#通过groups方法返回一个由模式各段组成的元组：
m = regex.match("wesm@bright.net")
print(m)
m.groups()

<re.Match object; span=(0, 15), match='wesm@bright.net'>


('wesm', 'bright', 'net')

m.groups() 在正则表达式中用于返回捕获组的元组，而不是进行分类。   
捕获组是用圆括号 () 包围的部分，表示你想要提取的特定部分。    
通过 groups()，你可以得到一个元组，里面包含匹配的每个组。


m.groups() 返回一个包含每个括号匹配的元组。  
'wesm' 对应第一个捕获组 ([A-Z0-9._%+-]+)，即邮箱用户名部分。  
'bright' 对应第二个捕获组 ([A-Z0-9.-]+)，即域名部分。  
'net' 对应第三个捕获组 ([A-Z]{2,4})，即顶级域名部分。  

正则表达式的 groups()：  

功能：用于提取字符串中匹配正则表达式的各个部分。每个用圆括号包裹的部分称为一个捕获组，groups() 方法会返回这些捕获组的匹配内容。  
作用场景：主要用于处理文本数据，匹配和分割字符串。  
示例：在邮箱地址中提取用户名、域名和顶级域名  


    


pd.Categorical (分类数据)：  

功能：将数据分成离散的类别，并且这些类别可以有顺序（有序类别）或无顺序（无序类别）。pd.Categorical 是 pandas 中的数据结构，用于表示带有有限类别的对象。  
作用场景：通常用于结构化数据分析，比如对年龄、评级等分类。它在节省内存和提高计算效率方面特别有用，尤其是处理重复值较多的列时。  
示例：将年龄数据分为不同的类别，如“青年”、“中年”、“老年”。  

In [491]:
#对带有分组的模式，findall会返回一个元素列表
regex.findall(text)

[('dave', 'google', 'com'),
 ('steve', 'gmail', 'com'),
 ('rob', 'gmail', 'com'),
 ('ryan', 'yahoo', 'com')]

In [493]:
#sub 还可以通过诸如\1、\2之类的特殊符号访问各匹配项中的分组。符号\1对应第一个匹配的组，\2对应第二个匹配的组，以此类推
print(regex.sub(r"Username: \1, Domain: \2, Suffix: \3", text))

Dave Username: dave, Domain: google, Suffix: com
Steve Username: steve, Domain: gmail, Suffix: com
Rob Username: rob, Domain: gmail, Suffix: com
Ryan Username: ryan, Domain: yahoo, Suffix: com


#### py中正则表达式  

![jupyter](7.5.png)

#### 7.4.3 pd中的字符串函数  
字符串中含有缺失数据：

In [501]:
data = {"Dave": "dave@google.com", "Steve": "steve@gmail.com",
        "Rob": "rob@gmail.com", "Wes": np.nan}
data = pd.Series(data)
data

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                  NaN
dtype: object

In [532]:
result = data.str[0]
result

Dave       d
Steve      s
Rob        r
Wes      NaN
dtype: object

In [503]:
data.isna()

Dave     False
Steve    False
Rob      False
Wes       True
dtype: bool

In [ ]:
通过data.map，所有字符串和正则表达式方法都能应用于（传入lambda表达式或其他函数）各个值，但是如果存在NA（空值）就会报错。
为了解决这个问题，Series为字符串操作提供了一些能够跳过并传播NA值的面向数组的方法。
通过Series的str属性即可访问这些方法。
例如，我们可以通过str.contains 检查各个电子邮件地址是否含有“gmail"：

In [505]:
data.str.contains("gmail")  #所有的字符串中是否包含gmail

Dave     False
Steve     True
Rob       True
Wes        NaN
dtype: object

注意，这个操作的结果中包含object数据类型。  
pandas提供了扩展类型，专门用于处理缺失数据中棘手的字符串、整数、布尔值数据：

In [515]:
data_as_string_ext = data.astype('string')   #修改Series数据类型为string，空值不会被修改
data_as_string_ext

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                 <NA>
dtype: string

In [534]:
result = data_as_string_ext.str[0]
result

Dave        d
Steve       s
Rob         r
Wes      <NA>
dtype: string

In [517]:
data_as_string_ext.str.contains("gmail")

Dave     False
Steve     True
Rob       True
Wes       <NA>
dtype: boolean

In [519]:
#pd字符串也可以使用正则表达式，并且可以和任意re配合进行
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"      #将符合正则表达式的邮箱分割输出
data.str.findall(pattern, flags=re.IGNORECASE)

Dave     [(dave, google, com)]
Steve    [(steve, gmail, com)]
Rob        [(rob, gmail, com)]
Wes                        NaN
dtype: object

In [ ]:
#多种方法实现向量化的元素获取操作，str.get,或者是在str属性上使用索引
matches = data.str.findall(pattern, flags=re.IGNORECASE)
matches

In [ ]:
matches = matches.str[0]   #输出每行的第一个元素，而不是字符串的第一个字符
matches

.str[0] 提取的是列表中的第一个元组，而不是 'Dave'、'Steve' 等字符串本身。  
如果你想要提取某个元组中的特定部分（如用户名 dave），你可以进一步使用 .str[n] 提取元组中的元素，比如 result.str[0] 会提取每个元组中的第一个元素（用户名）。

In [561]:
matches.str.get(1)  #这是输出每行中的列表或元组中的第二个元素

Dave     google
Steve     gmail
Rob       gmail
Wes         NaN
dtype: object

In [563]:
# 在str上用索引提取，可以实现字符串截取
data.str[:5]

Dave     dave@
Steve    steve
Rob      rob@g
Wes        NaN
dtype: object

In [565]:
# str.extract方法可以用DataFrame的形式返回正则表达式获取的分组：
data.str.extract(pattern, flags=re.IGNORECASE)

,0,1,2
Dave,dave,google,com
Steve,steve,gmail,com
Rob,rob,gmail,com
Wes,NaN,NaN,NaN


str.extract()会将捕获的每个组都转化为列以DF形式输出，正则表达式中每个括号就是一个组别

#### pd的字符串方法：  

![jupyter](7.6.png)
![jupyter](7.7.png)

### 7.5 分类数据  
利用pd提高性能，降低内存占用  

In [569]:
# 重复值处理
values = pd.Series(['apple', 'orange', 'apple', 'apple'] * 2)
values

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
dtype: object

In [571]:
pd.unique(values)

array(['apple', 'orange'], dtype=object)

In [573]:
pd.value_counts(values)

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_5224\3297668723.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(values)


apple     6
orange    2
Name: count, dtype: int64

In [579]:
#即将被优化的写法，需要用新的写法
pd.Series(values).value_counts()

apple     6
orange    2
Name: count, dtype: int64

许多数据系统（数据仓库、统计计算或其他用途）都发展出了专门的方法来表征重复值数据，以进行更高效的存储和计算。  
在数据仓库中，一种好的实践方法是使用包含不同值的维度表，将主要观测值存储为引用维度表的整数键  

人话：  使用维度表表示重复值数据，提高存储效率  

![jupyter](7.8.png)

In [582]:
values = pd.Series([0, 1, 0, 0] * 2)
dim = pd.Series(['apple', 'orange'])
values

0    0
1    1
2    0
3    0
4    0
5    1
6    0
7    0
dtype: int64

In [584]:
dim

0     apple
1    orange
dtype: object

In [607]:
dim.take([0, 1])

0     apple
1    orange
dtype: object

In [593]:
# 使用take方法存储原始的字符串Series
dim.take(values)     #基于values中的值从dim中提取相应元素

0     apple
1    orange
0     apple
0     apple
0     apple
1    orange
0     apple
0     apple
dtype: object

take() 本身并不会创建一个新的 Series，但它的作用是基于指定的索引位置从已有的 Series 中提取值，返回一个新的 Series。   

将values中的数值传入dim.take(),通过take函数组建成一个新的Series，节省了重复存储apple和orange的存储空间

1.这种用整数表示的方法称为分类表示法或字典编码表示法。  
2.不同值的数组称为数据的分类、字典或层级。  
3.分类表示可以在进行分析时显著提升性能。  
4.你也可以在保持编码不变的情况下，对分类进行转换。  

一些相对简单的转换示例包括：  
1.重命名分类。  
2.加人一个新的分类，不改变已经存在的分类的顺序或位置。  

#### pd的分类扩展空间  
pd有一个特殊的分类类型，用于保存使用整数分类表示法（或编码）的数据。这种
对于大量相似值的数据压缩技术，可以显著提升性能并降低内存占用，尤其是对于字符
串数据。

In [619]:
fruits = ['apple', 'orange', 'apple', 'apple'] * 2

N = len(fruits)

rng = np.random.default_rng(seed=12138)
df = pd.DataFrame({'fruit': fruits,    #水果名称列furuit 名称来自fruits列表
                   'basket_id': np.arange(N),    #生成序列
                   'count': rng.integers(3, 15, size=N),   #随机生成每个篮子中水果的数量区间[3，15]
                   'weight': rng.uniform(0, 4, size=N)},   #随机生成水果的重量区间[0,4]
                  columns=['basket_id', 'fruit', 'count', 'weight'])
df

,basket_id,fruit,count,weight
0,0,apple,9,1.945692
1,1,orange,6,3.675537
2,2,apple,5,1.336686
3,3,apple,5,3.023254
4,4,apple,3,1.374853
5,5,orange,11,3.213282
6,6,apple,13,0.917823
7,7,apple,12,1.833947


In [631]:
#这里，df['fruit']是一个Python字符串对象的数组。
#可以将其转换为分类数据的形式：
fruit_cat = df['fruit'].astype('category')
fruit_cat    #是 df 中fruit列 转化为 pd 中的类别数据类型，包含了原数据以及类别表。
#它仍然表现为与原 Series 相似的 Series 对象，但内部存储了对这些值的类别化信息。

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

In [623]:
#fruit_cat的值就变成了pandas.Categorical实例，可以访问其属性 .array进行确认：
c = fruit_cat.array
type(c)

pandas.core.arrays.categorical.Categorical

In [625]:
# Categorical对象有categories和codes两个属性：
c.categories

Index(['apple', 'orange'], dtype='object')

categories 是一个存储唯一类别值的列表，表示所有不同的类别（分类）是什么。  
例如，如果你有水果种类 ['apple', 'orange']，那么这些就是你的 categories。

In [627]:
c.codes

array([0, 1, 0, 0, 0, 1, 0, 0], dtype=int8)

codes 是原始数据中每个值对应的整数编码。  
类别的编码从 0 开始，表示每个类别在 categories 中的位置。  
例如，'apple' 可能会编码为0，而 'orange' 编码为 1。  

In [636]:
# 快速得到编码和分类的映射：
dict(enumerate(c.categories))

{0: 'apple', 1: 'orange'}

In [640]:
# 将 DataFrame 中的 fruit 列转换为类别数据类型，数据仍然和原来一样，但它们内部通过类别编码来表示，具有更高的存储和计算效率。
df['fruit'] = df['fruit'].astype('category')
df['fruit']

0     apple
1    orange
2     apple
3     apple
4     apple
5    orange
6     apple
7     apple
Name: fruit, dtype: category
Categories (2, object): ['apple', 'orange']

In [644]:
# 利用py序列直接创建pd.categorical
my_categories = pd.Categorical(['foo', 'bar', 'baz', 'foo', 'bar'])
my_categories

['foo', 'bar', 'baz', 'foo', 'bar']
Categories (3, object): ['bar', 'baz', 'foo']

In [648]:
# 已经得到分类编码，可以使用from_codes构造器：
categories = ['foo', 'bar', 'baz']    #分类标签
codes = [0, 1, 2, 0, 0, 1]    #索引值，快速分类数据 
my_cats_2 = pd.Categorical.from_codes(codes, categories)     #通过from_codes将编码和类别一一对应
my_cats_2

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo', 'bar', 'baz']

除非显式指定，否则分类转换不会指定分类的顺序。  
因此取决于输人数据的顺序，categories数组的顺序就会不同。  

In [657]:
#当使用from_codes或其他构造器时，给分类指定一个有意义的顺序：
ordered_cat = pd.Categorical.from_codes(codes, categories,
                                        ordered=True)
ordered_cat

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

Categories 的顺序，决定映射的顺序，['foo' < 'bar' < 'baz'] 代表了 foo 为 0，bar 为 1以此类推  
类别一旦调用函数生成后，就不好改变，所以一般都是指定类别顺序再去对应生成类别表

In [664]:
#无序的分类实例可以通过as_ordered排序，将无序转化为有序
my_cats_2.as_ordered()

['foo', 'bar', 'baz', 'foo', 'foo', 'bar']
Categories (3, object): ['foo' < 'bar' < 'baz']

通过将DF转化为类别数据类型，将字符串转化为数字，0~9可以快速提高效率，并且分类数据可以不是字符串，可以是其他任意的不可变类型

#### 7.5.3 用Categorical 对象进行计算  
分箱函数pd.cut工作原理：

In [668]:
rng = np.random.default_rng(seed=12138)     #default_rng使用最新算法生成的随机数随机性更强
draws = rng.standard_normal(1000)    #随机正态分布数
draws[:5]    #取前五个

array([-0.42362502,  0.58177077, -0.78045105,  0.64427673, -2.12222824])

In [672]:
# 计算该数据的四分位分箱，并提取统计信息
bins = pd.cut(draws, 4)    # 使用 pd.cut 将数据分割成 4 个区间
bins    #输出这四个区间的范围

[(-1.407, 0.0355], (0.0355, 1.478], (-1.407, 0.0355], (0.0355, 1.478], (-2.854, -1.407], ..., (0.0355, 1.478], (-1.407, 0.0355], (-1.407, 0.0355], (1.478, 2.92], (0.0355, 1.478]]
Length: 1000
Categories (4, interval[float64, right]): [(-2.854, -1.407] < (-1.407, 0.0355] < (0.0355, 1.478] < (1.478, 2.92]]

In [684]:
# qcut等频区间，每个区间分配一个标签
bins = pd.qcut(draws, 4, labels=['Q1', 'Q2', 'Q3', 'Q4']) 
print(pd.Series(bins).value_counts())
bins

Q1    250
Q2    250
Q3    250
Q4    250
Name: count, dtype: int64


['Q2', 'Q3', 'Q1', 'Q3', 'Q1', ..., 'Q3', 'Q2', 'Q2', 'Q4', 'Q3']
Length: 1000
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

In [686]:
# 带有标签的bins分类不包含数据分箱的边界信息，因此可以用groupby提取一些汇总统计信息：
bins = pd.Series(bins, name='quartile')
result = (pd.Series(draws).groupby(bins).agg(['count', 'min', 'max']).reset_index())
result

C:\Users\杜苏苏\AppData\Local\Temp\ipykernel_5224\3989752857.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result = (pd.Series(draws).groupby(bins).agg(['count', 'min', 'max']).reset_index())


,quartile,count,min,max
0,Q1,250,-2.848590,-0.772053
1,Q2,250,-0.767117,-0.039799
2,Q3,250,-0.039608,0.657397
3,Q4,250,0.661934,2.919672


pd.Series(bins, name='quartile') 将 bins 转换为 Series 对象，并命名为 'quartile'。  

pd.Series(draws).groupby(bins)：根据 bins 列对 draws 进行分组。  


.agg(['count', 'min', 'max'])：对每个分组计算 count、min和 max。  


.reset_index()：重置索引，以便将结果转换为 DataFrame 格式，方便查看和使用。

agg函数用于数据的聚合

In [699]:
#报的错误是表示 observed 参数将影响 groupby 聚合的结果。
#通过设置obeserved参数避免警告
bins = pd.Series(bins, name='quartile')    #保存分组的原始分类信息，并按照指定顺序排序
result = (pd.Series(draws).groupby(bins, observed=False).agg(['count', 'min', 'max']).reset_index())    #统计每个分组的信息
result

,quartile,count,min,max
0,Q1,250,-2.848590,-0.772053
1,Q2,250,-0.767117,-0.039799
2,Q3,250,-0.039608,0.657397
3,Q4,250,0.661934,2.919672


observed=False：计算所有可能的分组值。  
observed=True：仅计算在数据中实际出现的分组值。  

In [697]:
# 结果中的'quartile’列保留了bins中原始的分类信息，包括排序：
result['quartile']

0    Q1
1    Q2
2    Q3
3    Q4
Name: quartile, dtype: category
Categories (4, object): ['Q1' < 'Q2' < 'Q3' < 'Q4']

分类信息保留：保留原始的分类信息有助于理解分组数据的上下文。  
例如，了解哪些数据点在同一个四分位区间中，或者分组统计结果如何与分类标签相关联。  
顺序保留：保持分类的顺序可以帮助你清楚地了解数据是如何分布的，以及各个分类区间的相对位置。  

In [706]:
# 使用分类提高性能：
N = 10_000_000
labels = pd.Series(['foo', 'bar', 'baz', 'qux'] * (N // 4))
# 将labels 转化为分类：
categories = labels.astype('category')   #分类类型将字符串转换为整数编码，这样的整数运算比字符串运算快得多，从而提升了计算效率。

# labels使用的内存：
print("label使用的内存", labels.memory_usage(deep=True))

# 分类使用的内存：
categories.memory_usage(deep=True)

label使用的内存 520000132


10000512

In [710]:
# 分类转换的代价：
%time _ = labels.astype('category')

CPU times: total: 344 ms
Wall time: 524 ms


使用分类对象进行GroupBy操作明显更快，这是因为底层的算法使用了整数编码的
数组，而不是字符串数组。

In [713]:
#比较 value_counts（）的性能，它的内部也使用GroupBy机制：
%timeit labels.value_counts()

393 ms ± 12.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [715]:
%timeit categories.value_counts()

35.9 ms ± 397 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### 7.5.4 分类方法  
快捷访问分类和编码方法：


In [731]:
s = pd.Series(['a', 'b', 'c', 'd'] * 2)
cat_s = s.astype('category')
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [720]:
# cat 访问器属性提供了对分类的访问方法：
cat_s.cat.codes

0    0
1    1
2    2
3    3
4    0
5    1
6    2
7    3
dtype: int8

In [722]:
cat_s.cat.categories

Index(['a', 'b', 'c', 'd'], dtype='object')

In [748]:
#假设我们知道这个数据的实际分类集合超出了数据中观察到的4个值
#可以使用set_categories方法进行修改：
actual_categories = ['a', 'b', 'c', 'd', 'e']
# 使用 set_categories 方法将分类设置为 actual_categories
cat_s2 = cat_s.cat.set_categories(actual_categories)
cat_s2

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (5, object): ['a', 'b', 'c', 'd', 'e']

cat_s.cat.set_categories(actual_categories)：这是 Pandas 提供的一种方法，用于将分类 Series 的类别重新定义为指定的 actual_categories 列表。  
actual_categories = ['a', 'b', 'c', 'd', 'e'] 表示新的类别，包括原先存在的类别 a, b, c, d，以及新加入的 e。  

类别顺序：set_categories 只会更新类别的列表顺序，而不会改变数据本身。如果某些类别在数据中不存在，它们会被列出来，但不会有对应的值。  


缺失值处理：如果 cat_s 中存在的某些值不在 actual_categories 中，那么这些值将被视为缺失值 NaN。  

In [746]:
# 类别中不包含列表中的值，会出现Na
#new_categories = ['a', 'b', 'c']
#cat_s3 = cat_s.cat.set_categories(new_categories)
#cat_s3

0      a
1      b
2      c
3    NaN
4      a
5      b
6      c
7    NaN
dtype: category
Categories (3, object): ['a', 'b', 'c']

虽然数据看起来没变，但新的分类将反映在使用它们的操作中。   
例如，如果存在的话，value_counts将遵循新的分类：

In [751]:
cat_s.value_counts()

a    2
b    2
c    2
d    2
Name: count, dtype: int64

In [753]:
cat_s2.value_counts()

a    2
b    2
c    2
d    2
e    0
Name: count, dtype: int64

在大型数据集中，分类经常作为节省内存和提高性能的快捷工具。  


大型DataFrame或Series在做完过滤之后，许多分类可能不会出现在数据中。  

In [761]:
cat_s.isin(['a', 'b'])   #检查 cat_s 中的每个元素是否在指定的类别列表 ['a', 'b'] 

0     True
1     True
2    False
3    False
4     True
5     True
6    False
7    False
dtype: bool

In [763]:
# 使用remove_unused_categories方法删除未观察到的分类：
cat_s4 = cat_s[cat_s.isin(['a', 'b'])]    # 若上一步为True，则输出，Fasle则不输出
cat_s4

0    a
1    b
4    a
5    b
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [765]:
cat_s4.cat.remove_unused_categories()

0    a
1    b
4    a
5    b
dtype: category
Categories (2, object): ['a', 'b']

pd 中Series的分类放法：

![jupyter](7.9.png)

#### 为建模创建虚拟变量  
当使用统计或机器学习工具时，通常会将分类数据转换为虚拟变量，也称为独热编码。  
这包括创建一个DataFrame，其中每个不同的分类都是一列；对于给定并出现的分类，
这些列用1表示，未出现则用0表示。

In [773]:
# 前部分有介绍利用的例子，转化为机器可以读取的语言
cat_s = pd.Series(['a', 'b', 'c', 'd'] * 2, dtype='category')
cat_s

0    a
1    b
2    c
3    d
4    a
5    b
6    c
7    d
dtype: category
Categories (4, object): ['a', 'b', 'c', 'd']

In [775]:
#pd.get_dummies函数可以将这个一维分类数据转换为包含虚拟变量的DataFrame：
pd.get_dummies(cat_s)

,a,b,c,d
0,True,False,False,False
1,False,True,False,False
2,False,False,True,False
3,False,False,False,True
4,True,False,False,False
5,False,True,False,False
6,False,False,True,False
7,False,False,False,True


In [777]:
pd.get_dummies(cat_s).astype(int)

,a,b,c,d
0,1,0,0,0
1,0,1,0,0
2,0,0,1,0
3,0,0,0,1
4,1,0,0,0
5,0,1,0,0
6,0,0,1,0
7,0,0,0,1


pd.get_dummies 进行独热编码，可以将分类数据转换为二进制的特征矩阵，这些特征矩阵可以直接用于机器学习算法。

列：每个列代表一个类别。  
值：在每一行中，对于某个类别的列，其值是 1，其他列的值是 0。  

为什么独热编码对机器学习有用？  


模型兼容性：许多机器学习算法无法处理非数值数据。将分类变量转换为数值形式（独热编码）使得这些算法能够处理这些数据。


避免顺序问题：独热编码不会引入类别之间的顺序或大小关系（这种关系在某些模型中可能会造成问题），因为每个类别都被视为一个独立的特征。


特征分离：每个类别都被编码为一个独立的特征，确保模型能够独立处理每个类别。

维度增加：对于具有很多类别的分类变量，独热编码会显著增加特征维度，这可能会导致计算成本的增加和模型复杂性的提高。 


稀疏矩阵：如果类别非常多，结果可能是一个稀疏矩阵。你可以使用稀疏矩阵来节省内存，例如使用 scipy.sparse。

### cat 用法汇总：  
1. categories: 获取或设置分类数据的类别。

2. codes: 获取分类数据的编码（即每个类别的整数表示）。

3. add_categories(categories): 向分类数据添加新的类别。

4. remove_categories(categories): 从分类数据中删除指定的类别。

5. set_categories(categories, ordered=False): 设置新的类别，并可以选择是否将其排序。

6. as_ordered(): 将分类数据设置为有序的。

7. as_unordered(): 将有序的分类数据设置为无序的。

8. get_dummies(): 将分类数据转换为独热编码（one-hot encoding）形式。

